# 03 — GomJau-Hogg Notation

The GomJau-Hogg notation describes uniform Euclidean tilings as short codes
like `6-3-3/r60/r(h5)`. Each code is a sequence of *stages* separated by `/`:

1. **Polygon placement** (stage 1): polygons separated by `-` (sequential
   placement) and `,` (around the same vertex). A `0` means "skip an
   attachment slot".
2. **Transforms** (stage 2 and beyond): rotations (`r…`) and mirrors (`m…`)
   that expand the seed configuration into a tiling.

See [Gómez-Jáuregui & Hogg, *Symmetry* 13(12), 2021](https://www.mdpi.com/2073-8994/13/12/2376).

This notebook covers:

- the `gjh()` function and the cached library of 89 known tilings,
- inspecting the intermediate `TilesetSpec`,
- compiling novel codes with the parser,
- filling a bounded region with `fill_domain`.

In [ ]:
import os
os.environ.setdefault('TQDM_DISABLE', '1')

import matplotlib
matplotlib.rcParams['figure.figsize'] = (5, 5)
import matplotlib.pyplot as plt
import numpy as np

import eucare as ec
from eucare import example_graphs
from eucare.rendering import multi_show
from eucare.gjh import gjh, gjh_spec, gjh_graph, GJH_CODES

## Looking up a known tiling

`gjh(code)` returns a list of prototiles you can hand to `from_tiles` exactly
like the named factories in `example_tilesets`. Codes in the cached library
load instantly; novel codes are compiled on demand.

In [ ]:
codes_to_show = ['3/m30/r(h2)', '6-3-6/m30/r(v4)', '12-3/m30/r(h3)']
Gs = [example_graphs.from_tiles(gjh(code), rings=3) for code in codes_to_show]
multi_show(Gs, titles=codes_to_show, face_inset=0.05, render_vertices=False)

## Browsing the cached library

`GJH_CODES` is the ordered list of all cached codes (regular → 1-uniform →
2-uniform → 3-uniform, matching the source paper).

In [ ]:
print(f"Library size: {len(GJH_CODES)}")
print(f"First five: {GJH_CODES[:5]}")
print(f"Last three:  {GJH_CODES[-3:]}")

## Inspecting the spec

`gjh_spec(code)` returns the declarative `TilesetSpec` underlying a tiling.
The keys are tile names (one per congruency class); each value lists the
neighbour of every edge in cyclic order.

In [ ]:
spec = gjh_spec('3-6/m30/r(c2)')  # the 3.6.3.6 (trihexagonal) tiling
for name, edges in spec.items():
    print(f"  {name}: {edges}")

## Filling a bounded region

`fill_domain` grows a tileset until it covers a given `Domain` rather than a
fixed number of rings. Useful for rendering exact-size images.

In [ ]:
tiles = gjh('6-3-3/r60/r(h5)')  # 3.3.3.3.6
G = example_graphs.fill_domain(tiles, example_graphs.RectangleDomain(12, 8))
G.show(face_inset=0.05, render_vertices=False)

## Compiling a novel code

For codes not in the cached library, `gjh()` falls back to the parser. The
intermediate expanded graph is also available via `gjh_graph()`, useful for
debugging new codes.

In [ ]:
G = gjh_graph('4-3,3-0,4,3/r/r(h2)/r(h18)', bbox_size=16)
G.show(face_inset=0.05, render_vertices=False)